# Demo interactiva del sistema RAG

Complemento visual de `python -m rag.main` (ver `README.md` para el diseño completo). Cada celda
corre un paso real del pipeline end-to-end (ingesta → retrieval con score → generación grounded)
y muestra su salida real, sin mocks.

**Requisitos**: `pip install -r requirements.txt` y `.env` configurado (ver `README.md`).

In [1]:
import json

from rag.chain import answer_question
from rag.ingest import ingest_documentos

D:\AI-ENGINEERING-3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Módulo de ingesta

Fragmenta `rag/docs/*.md` con el splitter jerárquico (encabezados Markdown, con
`RecursiveCharacterTextSplitter` como fallback dentro de secciones largas) y persiste en ChromaDB.
Si la colección ya existe, no vuelve a indexar.

In [2]:
vectorstore = ingest_documentos()
print(f"Fragmentos en la colección: {vectorstore._collection.count()}")

Fragmentos en la colección: 24


## 2. Retrieval con score de similitud

Antes de pasar por el LLM: se ve qué fragmentos trae el retriever y con qué similitud real
(`1 - distancia`; la colección usa `hnsw:space="cosine"`, ver README).

In [3]:
pregunta_ejemplo = "¿Qué componente es el cuello de botella histórico del sistema bajo carga alta, y por qué?"

docs_con_distancia = await vectorstore.asimilarity_search_with_score(pregunta_ejemplo, k=4)
for doc, distancia in docs_con_distancia:
    print(f"{doc.metadata['source']:30s} similitud={1 - distancia:.2%}")

arquitectura_sistema.md        similitud=54.26%
monitoreo_alertas.md           similitud=21.45%
monitoreo_alertas.md           similitud=20.03%
monitoreo_alertas.md           similitud=16.51%


## 3. Cadena LCEL completa (retrieval + generación grounded)

`answer_question()` es async de punta a punta: retrieval → prompt "filtro de veracidad" →
`PydanticOutputParser` → validación (`_validar_salida`, con retry acotado). Se reutiliza el
`vectorstore` ya creado, en vez de reabrir la conexión a Chroma en cada llamada.

In [4]:
resultado = await answer_question(pregunta_ejemplo, vectorstore=vectorstore)
print(json.dumps(resultado.model_dump(), indent=2, ensure_ascii=False))

{
  "pregunta": "¿Qué componente es el cuello de botella histórico del sistema bajo carga alta, y por qué?",
  "respuesta": "El cuello de botella histórico del sistema bajo carga alta es el **pool de conexiones a PostgreSQL**. Específicamente, bajo más de 500 pedidos concurrentes, el límite de 100 conexiones de PgBouncer se satura, lo que causa que las requests nuevas queden esperando una conexión libre, generando timeouts intermitentes en el endpoint `/v1/pedidos`. Este es el incidente más frecuente registrado en el runbook de incidentes.",
  "contexto_encontrado": true,
  "fuentes": [
    "arquitectura_sistema.md",
    "monitoreo_alertas.md"
  ]
}


## 4. Fidelidad técnica: datos puntuales preservados, no parafraseados

Hallazgo real documentado en el README: en una primera versión del prompt, el modelo encontraba
el contexto correcto pero al redactar perdía datos operativos puntuales (ej. "réplicas de
PgBouncer", "150") en una paráfrasis genérica ("escalar el sistema"). El prompt actual instruye
explícitamente a preservarlos tal cual aparecen en el CONTEXTO.

*Nota*: como con cualquier LLM, esto reduce mucho el problema pero no lo garantiza al 100% en
cada corrida (Claude no es perfectamente determinístico incluso con `temperature=0`) -- por eso
la celda siguiente reporta el resultado en vez de asumirlo con un `assert` que podría fallar.

In [5]:
pregunta_tecnica = "¿Qué pasos hay que seguir para resolver un agotamiento del pool de conexiones a PostgreSQL?"

resultado_tecnico = await answer_question(pregunta_tecnica, vectorstore=vectorstore)
print(resultado_tecnico.respuesta)

datos_puntuales = ["réplicas de PgBouncer", "150"]
for dato in datos_puntuales:
    presente = dato in resultado_tecnico.respuesta
    print(f"{'✅' if presente else '⚠️ '} '{dato}' {'se conservó' if presente else 'NO aparece'} textualmente.")

Según el runbook de incidentes, los pasos para resolver un agotamiento del pool de conexiones a PostgreSQL son:

1. Verificar en el dashboard de PgBouncer cuántas conexiones están activas vs. el límite (100).

2. Si hay una query "colgada" reteniendo conexiones, identificarla con:
   ```
   SELECT * FROM pg_stat_activity WHERE state = 'active' ORDER BY query_start;
   ```
   y, si corresponde, cancelarla con `pg_cancel_backend(pid)`.

3. Si el problema es puramente de volumen (no hay queries colgadas), escalar horizontalmente el número de réplicas de PgBouncer o subir el límite de conexiones.
✅ 'réplicas de PgBouncer' se conservó textualmente.
⚠️  '150' NO aparece textualmente.


## 5. Caso trampa: pregunta fuera de todo el contexto disponible

El sistema debe reconocer que no tiene la información, en vez de inventar una respuesta plausible.

In [6]:
pregunta_trampa = "¿Cuál es la política de vacaciones del equipo de guardia?"

resultado_trampa = await answer_question(pregunta_trampa, vectorstore=vectorstore)
print(json.dumps(resultado_trampa.model_dump(), indent=2, ensure_ascii=False))

assert resultado_trampa.contexto_encontrado is False
assert resultado_trampa.fuentes == []
print("\nOK: 'No lo sé' en vez de alucinar, con fuentes vacías.")

{
  "pregunta": "¿Cuál es la política de vacaciones del equipo de guardia?",
  "respuesta": "No lo sé, no tengo información sobre eso en el contexto disponible.",
  "contexto_encontrado": false,
  "fuentes": []
}

OK: 'No lo sé' en vez de alucinar, con fuentes vacías.


---

Diseño completo, tests, y más hallazgos reales (chunking, embeddings, métrica de distancia) en
`README.md`. La suite de tests automatizados (mockeada, sin red) está en `rag/tests/` —
correr con `pytest rag/tests/ -v`.